# Optuna v4 — sin LINE (75mil, enero–junio 2025)

**Motivación**: el estudio de ablación sobre 75mil demostró que LINE no aporta nada (+0.04pp al quitarlo,
importancia por gradiente=0.0052). Este estudio busca los mejores hiperparámetros del modelo **sin LINE**,
para determinar si hay configuraciones mejores que Trial #57 una vez eliminado el módulo inútil.

**Diseño**:
- Dataset: `75mil_tarjetas_enero_junio_2025`
- Split: Fold 1 — train=ene–mar / test=abr (split_date=2025-04-01, seed=42)
- Épocas proxy: 25 (vs 7 en v3) — más fiables dado que GPU hace cada trial en ~10 min
- Objetivo: maximizar Top-1 accuracy en test
- Storage: SQLite persistente — reanudar con `load_if_exists=True`

**Fijados** (evidencia sólida de estudios v2/v3/75mil + ablación):

| Parámetro | Valor | Evidencia |
|-----------|-------|-----------|
| `ablation` | `{'line'}` | ablación: +0.04pp al quitar LINE |
| `optimizer` | `adamw` | consistente todos los estudios |
| `norm` | `None` | fANOVA 92% importancia en v2 |
| `n_heads` | `2` | consistente v3+75mil |
| `cnn_channels` | `32` | fijado en 75mil |
| `d_t` | `8` | fijado en 75mil |
| `kernel_size` | `3` | domina en top-v2, top-v3 |
| `lda_log_counts` | `True` | crítico 75mil (0.686 vs 0.577 sin él) |
| `lda_min_station_trips` | `0` | 75mil: filtrar perjudica |
| `lda_use_periods` | `False` | consistente todos los estudios |
| `lda_token_granularity` | `'temporal'` | consistente todos los estudios |
| `lr_scheduler` | `True` | 15/15 top-v2 |
| `include_prox` | `True` | ablación: −0.86pp al quitarlo |
| `include_neighborhood_crowd` | `True` | ablación: −0.44pp al quitarlo |
| `td_mode` | `'context'` | paper + moda top-v2 |

**Búsqueda**: `n_topics`, `lda_alpha`, `lda_beta`, `lda_min_df`, `lr`, `weight_decay`, `n_weeks`

In [ ]:
import sys, json
from pathlib import Path
from datetime import date

import optuna
import torch

_estudios = Path('/home/jupyter-lquijano/david/estudios')
_mdlf_dir = _estudios / 'MDLF'
sys.path.insert(0, str(_estudios))

from MDLF.pipeline_mdlf import PipelineMDLF

optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'Optuna  : {optuna.__version__}')
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ── Configuración ────────────────────────────────────────────────────────────

STUDY_NAME   = 'mdlf_75mil_v4_no_line'
DB_PATH      = _mdlf_dir / 'runs' / f'{STUDY_NAME}.db'
ALMACEN      = '75mil_tarjetas_enero_junio_2025'
N_EPOCHS     = 25
N_TRIALS     = 200
TIMEOUT_SEC  = 86400   # 24h

# Fold 1: train=ene–mar / test=abr
SPLIT_DATE   = '2025-04-01'
RANDOM_STATE = 42

# LINE parquet existente — se pasa al constructor pero nunca se carga (ablation={'line'})
LINE_AP_PATH = _mdlf_dir / 'LINE' / 'output' / 'line_cercanias_ap.parquet'

_descomp = _estudios / 'descomprimido'
TTP_PATHS = [str(_descomp / f'TTP_PERIODO_0{i}_2025.csv') for i in range(1, 7)]

OD_MATRICES_SUBDIRS = [
    'validaciones_3_7_febrero', 'validaciones_8_9_febrero',
    'validaciones_10_16_febrero', 'validaciones_17_23_febrero',
    'validaciones_24febrero_2marzo',
]

HOLIDAY_DATES = frozenset([
    date(2025, 1, 1), date(2025, 1, 6), date(2025, 4, 17), date(2025, 4, 18),
    date(2025, 5, 1), date(2025, 5, 2), date(2025, 5, 15),
])

# Parámetros fijos
FIXED = dict(
    ablation              = {'line'},
    optimizer             = 'adamw',
    norm                  = None,
    n_heads               = 2,
    cnn_channels          = 32,
    d_t                   = 8,
    kernel_size           = 3,
    lda_log_counts        = True,
    lda_min_station_trips = 0,
    lda_use_periods       = False,
    lda_token_granularity = 'temporal',
    lr_scheduler          = True,
    include_prox          = True,
    include_neighborhood_crowd = True,
    td_mode               = 'context',
)

print(f'Study   : {STUDY_NAME}')
print(f'DB      : {DB_PATH}')
print(f'Almacén : {ALMACEN}')
print(f'Split   : train < {SPLIT_DATE} | seed={RANDOM_STATE}')
print(f'Épocas  : {N_EPOCHS}  |  Trials: {N_TRIALS}  |  Timeout: {TIMEOUT_SEC//3600}h')

In [ ]:
# ── Función objetivo ─────────────────────────────────────────────────────────

def objective(trial: optuna.Trial) -> float:
    pipe = None
    try:
        # ── Espacio de búsqueda ───────────────────────────────────────────────
        # LDA
        n_topics  = trial.suggest_categorical('n_topics',  [32, 48, 64, 96])
        lda_alpha = trial.suggest_float('lda_alpha', 0.001, 1.0,  log=True)
        lda_beta  = trial.suggest_float('lda_beta',  0.001, 0.1,  log=True)   # nunca None: 75mil lo requiere
        lda_min_df = trial.suggest_int('lda_min_df', 1, 10)

        # Entrenamiento
        lr           = trial.suggest_float('lr',           1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)

        # Features
        n_weeks = trial.suggest_int('n_weeks', 3, 7)

        # ── Pipeline ──────────────────────────────────────────────────────────
        pipe = PipelineMDLF(
            nombre_almacen        = ALMACEN,
            ttp_paths             = TTP_PATHS,
            od_matrices_subdirs   = OD_MATRICES_SUBDIRS,
            line_ap_path          = LINE_AP_PATH,   # no se carga (LINE ablado)
            split_date            = SPLIT_DATE,
            holiday_dates         = HOLIDAY_DATES,
            random_state          = RANDOM_STATE,
            n_epochs              = N_EPOCHS,
            # LDA
            n_topics_cercanias    = n_topics,
            lda_alpha             = lda_alpha,
            lda_beta              = lda_beta,
            lda_min_df            = lda_min_df,
            # Entrenamiento
            lr                    = lr,
            weight_decay          = weight_decay,
            # Features
            n_weeks               = n_weeks,
            # Fijos
            **FIXED,
        )

        pipe.run(run_name=f'v4_no_line_t{trial.number:03d}')

        trial.set_user_attr('top5', pipe.test_accuracy5)
        return pipe.test_accuracy

    except Exception as e:
        # El trial falla pero el estudio continúa
        print(f'  [Trial {trial.number}] ERROR: {e}')
        raise optuna.exceptions.TrialPruned()

    finally:
        # Liberar GPU entre trials — crítico para evitar OOM acumulado
        if pipe is not None:
            try:
                pipe.model = pipe.model.cpu()
            except Exception:
                pass
            del pipe
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
# ── Crear / reanudar estudio ─────────────────────────────────────────────────

study = optuna.create_study(
    study_name    = STUDY_NAME,
    direction     = 'maximize',
    sampler       = optuna.samplers.TPESampler(seed=42, n_startup_trials=20),
    storage       = f'sqlite:///{DB_PATH}',
    load_if_exists= True,
)

trials_done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
print(f'Study   : {STUDY_NAME}')
print(f'DB      : {DB_PATH}')
print(f'Trials completados : {trials_done} / {N_TRIALS}')
if trials_done > 0:
    print(f'Mejor hasta ahora  : {study.best_value:.4f}  (trial #{study.best_trial.number})')

In [ ]:
# ── Ejecutar optimización ────────────────────────────────────────────────────
# Para en el primero de: N_TRIALS trials completados ó TIMEOUT_SEC segundos

study.optimize(
    objective,
    n_trials         = N_TRIALS,
    timeout          = TIMEOUT_SEC,
    show_progress_bar= True,
)

trials_done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
print(f'\n✓ Optimización terminada')
print(f'  Trials completados : {trials_done}')
print(f'  Mejor Top-1        : {study.best_value:.4f}')
print(f'  Trial nº           : {study.best_trial.number}')
print(f'  Mejores params     :')
for k, v in study.best_trial.params.items():
    print(f'    {k:<20} {v}')

In [ ]:
# ── Resultados y análisis ────────────────────────────────────────────────────
import pandas as pd

df = study.trials_dataframe()
df_ok = df[df['state'] == 'COMPLETE'].copy()

param_cols   = [c for c in df_ok.columns if c.startswith('params_')]
rename_map   = {c: c.replace('params_', '') for c in param_cols}
rename_map.update({'value': 'top1', 'number': 'trial'})

df_top = (
    df_ok
    .sort_values('value', ascending=False)
    .head(20)
    [['number', 'value'] + param_cols]
    .rename(columns=rename_map)
    .reset_index(drop=True)
)

print(f'Trials completados : {len(df_ok)}')
print(f'Mejor  top-1       : {df_ok["value"].max():.4f}')
print(f'Mediana top-1      : {df_ok["value"].median():.4f}')
print(f'P75    top-1       : {df_ok["value"].quantile(0.75):.4f}')
print()

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.5f}' if abs(x) < 10 else f'{x:.2f}')
display(df_top)

In [ ]:
# ── Importancia de hiperparámetros (fANOVA) ──────────────────────────────────
try:
    importances = optuna.importance.get_param_importances(study)
    print('Importancia de hiperparámetros (fANOVA):')
    for param, imp in importances.items():
        bar = '█' * int(imp * 60)
        print(f'  {param:<20} {imp:.4f}  {bar}')
except Exception as e:
    print(f'No disponible aún (mínimo ~20 trials completos): {e}')

In [ ]:
# ── Guardar mejor configuración en JSON ──────────────────────────────────────
best = study.best_trial

out = {
    'study_name'   : STUDY_NAME,
    'best_trial'   : best.number,
    'top1_fold1'   : best.value,
    'top5_fold1'   : best.user_attrs.get('top5'),
    'n_epochs'     : N_EPOCHS,
    'split_date'   : SPLIT_DATE,
    'best_params'  : best.params,
    'fixed_params' : {k: str(v) for k, v in FIXED.items()},
}

json_path = _mdlf_dir / 'runs' / f'{STUDY_NAME}_best.json'
with open(json_path, 'w') as f:
    json.dump(out, f, indent=2, default=str)

print(f'Guardado: {json_path}')
print(json.dumps(out, indent=2, default=str))